# 编程模型概述

## 学习目标

完成本节学习后，你应该能够：

- 理解 AI Core 中各个核心单元的基本角色；
- 理解 SIMD 编程模型的核心思想：外层多核 SPMD 分片，内层单指令批量处理数据；
- 理解 SIMT 编程模型的核心思想：Grid-Block-Thread 线程层级和线程索引映射；
- 根据访存模式、分支复杂度和目标硬件能力，初步判断一个算子更适合 SIMD 还是 SIMT。


## 1. 异构系统 与 AI Core硬件基础

基于昇腾处理器的应用程序通常包含主机（或称Host）侧代码和 设备侧（或称Device、NPU）代码。

- **Host 侧代码**运行在 CPU 上，负责设备资源管理、Host Memory 与 Device Memory 数据搬运等，并向Device侧下发计算任务。
- **Device 侧代码**运行在 NPU 上，通过任务调度器调度，在 AI Core 上执行真正的算子计算逻辑。

<img src="./images/kernel_schedule_flow.png" alt="Kernel 调度示意图" width="520px">

AI Core 是昇腾 NPU 上执行算子计算的核心单元。为了便于理解 SIMD 和 SIMT 编程模型，可以先把 AI Core 抽象为三类组件。

| 组件类别 | 典型组件 | 在算子中的作用 |
| --- | --- | --- |
| 计算单元 | Scalar、Vector、Cube | Scalar 负责控制逻辑和指令发射；Vector 负责向量计算；Cube 负责矩阵计算。 |
| 存储单元 | Global Memory、Local Memory | Global Memory 是 Device 侧全局存储；Local Memory 是 AI Core 片内高速存储。 |
| 搬运单元 | DMA | 在 Global Memory 与 Local Memory、不同 Local Memory 层级之间搬运数据。 |

理解AI Core硬件架构是开发高性能算子的核心前提，例如：明确数据需主动搬运至本地存储，才能充分发挥SIMD高带宽算力优势；掌握向量处理单元的硬件特性，才能主动将计算逻辑向量化，最大化发挥硬件峰值性能，避免算力浪费。


## 2. SIMD 编程模型

SIMD 是 Single Instruction Multiple Data 的缩写，含义是“一条指令同时处理多份数据”。SPMD 是 Single Program Multiple Data 的缩写，含义是“同一份程序处理不同的数据分片”。在 Ascend C 算子开发中，SIMD 通常可以从两个层次理解。

- **外层多核 SPMD**：多个 AI Core 运行同一份计算逻辑，每个 AI Core 根据自己的逻辑核索引处理不同数据分片。
- **内层单核 SIMD**：单个 AI Core 内部通过 Vector 或 Cube 等计算单元，用一条向量/矩阵指令批量处理一组规整数据。

SIMD 算子不是“一个核处理全部数据”，而是先把一大块数据切成多个数据分片，再让每个 AI Core 在片内使用 SIMD 指令完成高吞吐计算。

![SIMD SPMD 分片模型](./images/simd_spmd_partition.svg)


### 2.1 SIMD 算子开发

SIMD 算子通常遵循“分块、搬入、计算、搬出”的数据流。

1. **Tiling 分块**：Host 侧逻辑把全局数据切成适合多个 AI Core 处理的数据片。
2. **数据搬入**：Device 侧计算逻辑将当前数据片从 Global Memory 搬到 Local Memory，例如 UB、L1、寄存器等片内资源。
3. **数据计算**：调用 Vector 或 Cube 相关能力，在片内对规整数据做批量计算。
4. **数据搬出**：把计算结果从片内存储写回 Global Memory。

这一模型适合连续访存、分支较少、数据形状规整的计算，例如向量逐元素计算、矩阵乘、卷积以及许多深度学习算子的高密度计算部分。


### 2.2 SIMD 适用场景

SIMD 主要适配数据密集、操作规整、无分支或分支极少的计算任务。典型场景包括：

- 图像像素处理，如灰度化、滤波、像素缩放；
- 音频信号分析，如降噪、信号预处理；
- 矩阵乘法、卷积等深度学习核心运算；
- 逐元素数学函数，如向量加减、乘除、指数/对数运算。


## 3. SIMT 编程模型

SIMT 是 Single Instruction Multiple Thread 的缩写，含义是“一条指令同时驱动多个线程”。SIMT 把并行粒度从“向量指令处理一批数据”转向“多个线程分别处理数据元素”。

SIMT 编程通常使用业界常见的三级线程层级。

- **Grid**：一次 Device 侧并行任务形成的线程块集合，可以通过内置变量 `gridDim` 表示启用的线程块个数。
- **Thread Block**：线程块，一个 Block 中包含多个线程，可以通过内置变量 `blockDim` 表示一个线程块启用的线程个数，通过内置变量 `blockIdx` 表示当前线程块索引。
- **Thread**：最小执行单位，每个线程有自己的线程索引内置变量 `threadIdx`，通常处理一个或少量数据元素。

![SIMT 三级线程层级](./images/03_02_simt_thread_hierarchy.svg)

注意，SIMT 当前仅在 Ascend 950PR/Ascend 950DT 上支持。


### 3.1 SIMT 算子开发

SIMT 算子通常可以按“线程划分、数据访问、线程计算、结果写回”的流程理解。

1. **线程划分**：将整体任务拆分为多个线程，使线程索引与数据索引对应，避免重复处理或遗漏数据。
2. **数据访问**：线程通过指针访问 Device Memory，硬件将所需数据加载到线程可使用的寄存器等资源中。
3. **线程计算**：每个线程执行相同程序，但可根据自己的索引和条件处理不同数据，适合表达分支、循环等复杂控制逻辑。
4. **结果写回**：线程将计算结果写回 Device Memory；如果涉及线程间协作，需要结合共享内存和同步机制保证结果正确。


### 3.2 SIMT 适用场景

SIMT 主要适配不规则数据访问、分支密集、稀疏计算等场景。典型场景包括：

- 深度学习中的稀疏算子，如稀疏卷积、稀疏矩阵运算；
- 带有复杂 `if-else` 分支的逐元素运算；
- 具有动态数据依赖的算法，如并行前缀和、排序网络。


## 4. SIMD 与 SIMT 对比


| 对比维度 | SIMD | SIMT |
| --- | --- | --- |
| 并行粒度 | 一条指令批量处理多份同构数据 | 一条指令驱动多个线程，每个线程处理自己的数据 |
| 编程模型 | 多 AI Core SPMD + 单核 SIMD 指令 | Grid-Block-Thread 线程层级 |
| 分支适配 | 更适合分支少、数据规整的高吞吐计算 | 更适合复杂分支、离散访存、不规则索引 |
| 计算单元 | Vector 单元和Cube 单元 | 支持 SIMT 的 Vector 单元 |
| 典型场景 | 图像像素处理、音频信号分析、矩阵乘法、卷积、逐元素数学函数 | 稀疏算子、复杂 `if-else` 分支逐元素运算、动态数据依赖算法 |
| 硬件支持 | 昇腾全系列 | 当前仅支持 Ascend 950PR/Ascend 950DT 芯片架构 |


## 5. 本节小结

本节从 Ascend C 算子开发的整体视角介绍了 SIMD 与 SIMT 两种 AI Core 编程模型。

- Ascend C 算子通常由 Host 侧负责资源、数据和调度，由 Device 侧 AI Core 执行计算；
- AI Core 的基本组成包括计算单元、存储单元和搬运单元，开发高性能算子时需要关注数据在 Global Memory、Local Memory 和计算单元之间的流动；
- SIMD 是主流高吞吐模型，适合数据密集、操作规整、无分支或分支极少的计算任务；
- SIMT 是线程级并行模型，适合不规则数据访问、分支密集、稀疏计算等场景；


## 6. 课后练习

请根据本节内容完成以下题目进行自测。

1. （判断题）SIMT 编程模型既支持向量计算，也可以直接替代 SIMD 完成 Cube 矩阵计算，因此实际开发中不需要再学习 SIMD。

2. （单选题）在 SIMT 一维线程映射 `global_id = blockIdx.x * blockDim.x + threadIdx.x` 中，`threadIdx.x` 表示什么？  
    A. 当前 Grid 中线程块的数量  
    B. 当前线程在线程块内的索引  
    C. 当前 AI Core 的 SIMD 分片编号  
    D. 当前并行任务的下发次数  

3. （多选题）以下哪些情况通常更适合优先考虑 SIMD？  
    A. 数据连续规整，多个元素执行相同计算  
    B. 分支较少，主要目标是高吞吐  
    C. 算子核心计算可以由 Vector 或 Cube 批量完成  
    D. 每个元素都需要访问完全不同的离散地址，并走大量不同分支  

4. （多选题）以下哪些属于 SIMT 编程模型更适合处理的场景？  
    A. 深度学习中的稀疏算子  
    B. 带有复杂 `if-else` 分支的逐元素运算  
    C. 具有动态数据依赖的算法  
    D. 数据连续、分支很少的规整矩阵乘  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.02_answer.txt
